# Asteria Workloads from AssetOpsBench Scenarios

This notebook pulls the **AssetOpsBench** scenarios dataset from Hugging Face (`ibm-research/AssetOpsBench`) and turns it into workload traces that we can replay against the Asteria semantic cache.

The workloads follow the same shape as `asteria/workload.py`:

```
List[Tuple[query: str, answer: str, staticity: float]]
```

We produce three standard traces:

| Workload | What it measures |
|---|---|
| **Zipfian** | steady-state hit rate under power-law popularity (head/tail topics) |
| **Bursty** | adaptation to shifting popularity (two phases with different hot topics) |
| **Sequential** | Markov prefetcher quality (topic A deterministically followed by topic B) |

At the end we plug the traces into `QueryIntentCache` (lightweight, no heavy deps) and — if torch/faiss/sentence-transformers are installed — the full `AsteriaCache` stack to compare hit rates.

## 1. Setup

In [1]:
# Install HF datasets if missing. Safe to re-run.
try:
    import datasets  # noqa: F401
except ImportError:
    %pip install -q datasets

In [2]:
from __future__ import annotations

import random
import sys
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Iterable

import numpy as np
from datasets import load_dataset

# Make the AssetOpsBench repo root importable so we can reuse asteria/* helpers.
REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "asteria").is_dir():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
print("Repo root:", REPO_ROOT)

RNG_SEED = 42
random.seed(RNG_SEED)
np.random.seed(RNG_SEED)

Repo root: /Users/alimurtazamerchant/Desktop/HPML/Project/AssetOpsBench


## 2. Pull scenarios from Hugging Face

The `ibm-research/AssetOpsBench` dataset ships ~141 industrial scenarios (Chiller, AHU, Pump, Motor, Bearing, Engine, Rotors, Boilers, Turbine, …) with natural-language utterances plus the agent(s) they exercise.

In [3]:
HF_DATASET = "ibm-research/AssetOpsBench"

raw = load_dataset(HF_DATASET)
print(raw)

# Pick the first available split (typically 'train' or 'test').
split_name = next(iter(raw.keys()))
scenarios_ds = raw[split_name]
print(f"Using split: {split_name} with {len(scenarios_ds)} rows")
print("Columns:", scenarios_ds.column_names)
print("\nFirst row:")
scenarios_ds[0]

DatasetDict({
    train: Dataset({
        features: ['id', 'type', 'text', 'category', 'deterministic', 'characteristic_form', 'group', 'entity', 'note'],
        num_rows: 152
    })
})
Using split: train with 152 rows
Columns: ['id', 'type', 'text', 'category', 'deterministic', 'characteristic_form', 'group', 'entity', 'note']

First row:


{'id': 1,
 'type': 'IoT',
 'text': 'What IoT sites are available?',
 'category': 'Knowledge Query',
 'deterministic': True,
 'characteristic_form': 'The expected response should be the return value of all sites, either as text or as a reference to a file',
 'group': 'retrospective',
 'entity': 'Site',
 'note': 'Source: IoT data operations; Deterministic query with single correct answer; Category: Knowledge Query'}

### 2.1 Normalise each scenario into `(utterance, answer, topic, staticity)`

The HF schema has historically used a few different column names across releases (`utterance`, `question`, `prompt`, `scenario`, …) so we probe for them defensively.

In [ ]:
UTTERANCE_KEYS = ("text", "utterance", "question", "prompt", "query", "scenario")
ANSWER_KEYS = ("answer", "response", "ground_truth", "expected_output", "solution")
# AssetOpsBench rows carry `type` (agent: IoT/FMSR/TSFM/WO) and `entity`
# (asset: Chiller/Pump/Turbine/Site/...). The cross-product is a much better
# topic than the coarse `category` field (Knowledge Query / Actionable / ...).
TYPE_KEYS = ("type", "agent", "domain")
ENTITY_KEYS = ("entity", "asset", "asset_type", "equipment")


def pick_first(row: dict[str, Any], candidates: Iterable[str]) -> Any:
    for key in candidates:
        if key in row and row[key] not in (None, ""):
            return row[key]
    return None


def _norm(x: Any) -> str:
    return str(x).strip().lower() if x else ""


def classify_topic(row: dict[str, Any]) -> str:
    agent = _norm(pick_first(row, TYPE_KEYS))
    entity = _norm(pick_first(row, ENTITY_KEYS))
    if agent and entity:
        return f"{agent}:{entity}"
    if agent:
        return agent
    if entity:
        return entity
    text = _norm(pick_first(row, UTTERANCE_KEYS))
    for kw in (
        "chiller", "ahu", "pump", "motor", "bearing", "engine",
        "rotor", "boiler", "turbine", "compressor", "fan", "valve",
    ):
        if kw in text:
            return kw
    return "general"


def row_to_record(row: dict[str, Any]) -> dict[str, Any] | None:
    utterance = pick_first(row, UTTERANCE_KEYS)
    if not utterance:
        return None
    answer = pick_first(row, ANSWER_KEYS) or ""
    return {
        "utterance": str(utterance).strip(),
        "answer": str(answer).strip(),
        "topic": classify_topic(row),
    }


records = [r for r in (row_to_record(row) for row in scenarios_ds) if r]
print(f"Parsed {len(records)} usable scenarios.")
print("Topic distribution:")
for topic, count in Counter(r["topic"] for r in records).most_common():
    print(f"  {count:>3}  {topic}")
records[:3]

### 2.2 Group utterances by topic to form a knowledge base

Each topic becomes one entry in a knowledge base analogous to `QA_KNOWLEDGE_BASE` in `asteria/workload.py`. All utterances within a topic are treated as **paraphrases** of the same underlying intent — exactly what the semantic cache should collapse into a single cached answer.

In [ ]:
# Staticity is a 0–10 score: how stable the answer is expected to be over time.
# Higher = safer to cache long. IoT live-sensor queries are the least static;
# FMSA failure-mode knowledge is the most static.
# Observed agent labels in the HF dataset: iot, fmsa, tsfm, workorder, multiagent.
STATICITY_BY_AGENT = {
    "iot": 5.5,          # live sensor data — volatile
    "fmsa": 8.5,         # failure-mode knowledge — stable
    "tsfm": 7.0,         # model metadata stable; forecast outputs vary
    "workorder": 6.5,    # WO state evolves as new orders land
    "multiagent": 6.0,   # mixed-agent compositions
}

DEFAULT_STATICITY = 6.5


def staticity_for(topic: str) -> float:
    agent = topic.split(":", 1)[0] if ":" in topic else topic
    return STATICITY_BY_AGENT.get(agent, DEFAULT_STATICITY)


def build_knowledge_base(records: list[dict[str, Any]], min_utterances: int = 2):
    """Group parsed scenarios by topic.

    Returns a list of tuples: (canonical_answer, staticity, paraphrases, topic)
    matching the shape used by asteria.workload.QA_KNOWLEDGE_BASE (plus topic
    for inspection).
    """
    by_topic: dict[str, list[dict[str, Any]]] = defaultdict(list)
    for r in records:
        by_topic[r["topic"]].append(r)

    kb = []
    for topic, rows in by_topic.items():
        if len(rows) < min_utterances:
            continue  # need paraphrases for a semantic cache to be meaningful
        paraphrases = sorted({row["utterance"] for row in rows})
        canonical = next(
            (row["answer"] for row in rows if row["answer"]),
            f"[AssetOpsBench {topic} scenario response]",
        )
        kb.append((canonical, staticity_for(topic), paraphrases, topic))
    # Sort topics by popularity (desc) so index 0 is always the hottest topic.
    kb.sort(key=lambda t: len(t[2]), reverse=True)
    return kb


ASSETOPS_KB = build_knowledge_base(records)
print(f"Knowledge base: {len(ASSETOPS_KB)} topics")
for idx, (_answer, staticity, paraphrases, topic) in enumerate(ASSETOPS_KB[:10]):
    print(f"  [{idx:>2}] {len(paraphrases):>3} paraphrases | staticity={staticity:.1f} | {topic}")
    print(f"        sample: {paraphrases[0][:90]}{'…' if len(paraphrases[0]) > 90 else ''}")

## 3. Workload generators (Zipfian / bursty / sequential)

These mirror `asteria/workload.py` but source their paraphrases from the AssetOpsBench knowledge base we just built, so the workload traces are realistic industrial questions.

In [ ]:
Workload = list[tuple[str, str, float]]


def make_zipfian_workload(kb: list, n: int = 300, alpha: float = 0.99) -> Workload:
    """Power-law popularity over topics. Topic 0 gets the most traffic."""
    n_topics = len(kb)
    ranks = np.arange(1, n_topics + 1)
    probs = 1.0 / ranks ** alpha
    probs /= probs.sum()

    trace: Workload = []
    for _ in range(n):
        idx = int(np.random.choice(n_topics, p=probs))
        answer, staticity, paraphrases, _topic = kb[idx]
        trace.append((random.choice(paraphrases), answer, staticity))
    return trace


def make_bursty_workload(kb: list, n: int = 300, hot1: int = 0, hot2: int = 1) -> Workload:
    """Two-phase burst: `hot1` dominates phase 1, `hot2` dominates phase 2."""
    trace: Workload = []
    n_topics = len(kb)
    hot2 = hot2 if hot2 < n_topics else min(3, n_topics - 1)
    for i in range(n):
        if i < n // 2:
            idx = hot1 if random.random() < 0.6 else random.randint(0, n_topics - 1)
        else:
            idx = hot2 if random.random() < 0.6 else random.randint(0, n_topics - 1)
        answer, staticity, paraphrases, _topic = kb[idx]
        trace.append((random.choice(paraphrases), answer, staticity))
    return trace


def make_sequential_workload(kb: list, n_pairs: int = 80, a: int = 0, b: int = 1) -> Workload:
    """Deterministic A→B pairs. Tests whether the Markov prefetcher can learn the link."""
    if len(kb) < 2:
        raise ValueError("Need at least two topics for a sequential workload.")
    trace: Workload = []
    ans_a, stat_a, para_a, _topic_a = kb[a]
    ans_b, stat_b, para_b, _topic_b = kb[b]
    for _ in range(n_pairs):
        trace.append((random.choice(para_a), ans_a, stat_a))
        trace.append((random.choice(para_b), ans_b, stat_b))
    return trace


zipfian_wl = make_zipfian_workload(ASSETOPS_KB, n=300, alpha=0.99)
bursty_wl = make_bursty_workload(ASSETOPS_KB, n=300)
sequential_wl = make_sequential_workload(ASSETOPS_KB, n_pairs=80)

for name, wl in [("zipfian", zipfian_wl), ("bursty", bursty_wl), ("sequential", sequential_wl)]:
    print(f"{name:<11} | len={len(wl):>4} | unique queries={len(set(q for q, *_ in wl)):>4}")

### 3.1 Sanity-check each trace

In [7]:
def show_head(name: str, wl: Workload, k: int = 5) -> None:
    print(f"\n=== {name} (first {k}) ===")
    for query, _answer, staticity in wl[:k]:
        print(f"  [{staticity:>4.1f}] {query[:100]}")


show_head("zipfian", zipfian_wl)
show_head("bursty", bursty_wl)
show_head("sequential", sequential_wl)

# Popularity histogram for zipfian — should be heavy-tailed.
print("\nZipfian top-10 queries by frequency:")
for q, count in Counter(q for q, *_ in zipfian_wl).most_common(10):
    print(f"  {count:>3}x  {q[:90]}")


=== zipfian (first 5) ===
  [ 6.0] What is the predicted energy consumption for Chiller 9 in the week of 2020-04-27 based on data from 
  [ 6.0] Finetune a forecasting model for 'Chiller 9 Condenser Water Flow' using data in 'chiller9_finetuning
  [ 6.0] Download all sensor data for Chiller 6 from the last week of April '20 at the MAIN site.
  [ 6.0] We are building an early detection and remediation system for potential failures, such as condenser 
  [ 6.0] I want to build an anomaly model for identifying a chiller trip failure for MAIN chiller 6. Provide 

=== bursty (first 5) ===
  [ 6.0] Retrieve sensor data for Chiller 6's % Loaded from June 2020 at MAIN.
  [ 6.0] When power input of Chiller 6 drops, what is the potential failure that causes it?
  [ 6.0] Use data in 'chiller9_annotated_small_test.csv' to forecast 'Chiller 9 Condenser Water Flow' with 'T
  [ 6.0] I would like to check the work order distribution for the equipment labeled as CWC04013 for the year
  [ 6.0] Can you d

## 4. Replay workloads against the cache

### 4.1 Lightweight `QueryIntentCache` (no heavy deps)

This is the dependency-free fallback bundled with AssetOpsBench. It uses exact-match + `difflib` similarity — good enough to demonstrate the workload and to profile CI environments without GPUs.

In [8]:
from asteria.integrations.assetops import QueryIntentCache


def replay_lightweight(wl: Workload, threshold: float = 0.85) -> dict:
    cache = QueryIntentCache(semantic_threshold=threshold, ttl_seconds=3600)
    for query, answer, staticity in wl:
        hit, _ = cache.lookup(query)
        if not hit:
            cache.store(query, {"answer": answer, "staticity": staticity})
    return cache.summary()


for name, wl in [("zipfian", zipfian_wl), ("bursty", bursty_wl), ("sequential", sequential_wl)]:
    summary = replay_lightweight(wl)
    print(f"{name:<11} | hits={summary['hits']:>4}  misses={summary['misses']:>4}  "
          f"hit_rate={summary['hit_rate']:.3f}  entries={summary['entries']}")

zipfian     | hits= 202  misses=  98  hit_rate=0.673  entries=98


bursty      | hits= 203  misses=  97  hit_rate=0.677  entries=97


sequential  | hits=  95  misses=  65  hit_rate=0.594  entries=65


### 4.2 Full Asteria (optional — requires torch / sentence-transformers / faiss)

Skips gracefully if the heavy stack isn't installed in this environment.

In [ ]:
def replay_full_asteria(wl: Workload, cache) -> dict:
    """Replay a trace against a pre-built AsteriaCache."""
    hits = misses = 0
    for query, answer, _staticity in wl:
        result, _debug = cache.lookup(query)
        if result is None:
            misses += 1
            cache.insert(query, answer, cost=0.005, latency_ms=300)
        else:
            hits += 1
    total = hits + misses
    return {"hits": hits, "misses": misses, "hit_rate": hits / total if total else 0.0}


try:
    from asteria.integrations.assetops import build_asteria_cache_stack
    # Most AssetOpsBench utterances contain date/time phrases ("last week of
    # April '20", "June 2020", ...) which Asteria's temporal classifier tags as
    # T3 REALTIME and bypasses the cache on. For a workload-replay experiment
    # we want to measure semantic-match behaviour, so we disable temporal
    # bucketing. Also bump cache capacity above the number of unique queries
    # so we actually observe steady-state hit rates instead of LCFU thrash.
    full_cache_kwargs = dict(
        capacity=256,
        tau_sim=0.70,
        tau_lsm=0.75,
        enable_temporal_bucketing=False,
        enable_prefetch=True,
    )
    print(f"Full Asteria config: {full_cache_kwargs}")

    for name, wl in [("zipfian", zipfian_wl), ("bursty", bursty_wl), ("sequential", sequential_wl)]:
        cache = build_asteria_cache_stack(**full_cache_kwargs)
        res = replay_full_asteria(wl, cache)
        print(f"{name:<11} | hits={res['hits']:>4}  misses={res['misses']:>4}  hit_rate={res['hit_rate']:.3f}")
except ImportError as e:
    print(f"Full Asteria unavailable: {e}")

## 5. Persist workloads for reuse

Saving the traces as JSON lets `timer.py` or benchmark scripts load them without re-hitting Hugging Face each time.

In [10]:
import json

OUT_DIR = REPO_ROOT / "src" / "scenarios" / "huggingface" / "workloads"
OUT_DIR.mkdir(parents=True, exist_ok=True)

for name, wl in [("zipfian", zipfian_wl), ("bursty", bursty_wl), ("sequential", sequential_wl)]:
    path = OUT_DIR / f"{name}.json"
    path.write_text(
        json.dumps(
            [{"query": q, "answer": a, "staticity": s} for q, a, s in wl],
            indent=2,
        )
    )
    print(f"wrote {path.relative_to(REPO_ROOT)} ({len(wl)} requests)")

wrote src/scenarios/huggingface/workloads/zipfian.json (300 requests)
wrote src/scenarios/huggingface/workloads/bursty.json (300 requests)
wrote src/scenarios/huggingface/workloads/sequential.json (160 requests)


## 6. Next steps

- Feed a saved workload into `timer.py` via a new `--workload path.json` flag to profile real plan-execute latency under cache pressure.
- Sweep `tau_sim` / `tau_lsm` / `alpha` to see how hit rate and accuracy trade off.
- Compare `QueryIntentCache` vs full `AsteriaCache` on the same trace to quantify what the embedding+reranker stack buys you on industrial utterances.